# 21.11 湖仓一体:Delta / Iceberg / Hudi / Lakehouse: Delta / Iceberg / Hudi

**中文**:数据存储长期有两个割裂的世界:**数据仓库(warehouse)**——有 ACID 事务、能改能删、schema 严格、查询快,但**贵、封闭、难扩展**(Snowflake、Redshift);**数据湖(data lake)**——把 Parquet 文件堆在便宜的对象存储(S3)上,**便宜、开放、无限扩展**,但**没有事务、不能安全更新/删除、没有版本、schema 混乱**(21.8 讲的裸 Parquet)。**湖仓一体(Lakehouse)** 是过去几年最重要的架构革命:*在廉价的数据湖(Parquet on S3)之上,加一层"事务日志/元数据层",就能获得数据仓库级别的 ACID 事务、更新删除、时间旅行、schema 管理*。实现这个魔法的是 **Delta Lake / Apache Iceberg / Apache Hudi** 这三种"表格式(table format)"。本节从零实现一个 mini-Lakehouse——用真实 Parquet 文件 + 一个事务日志,亲手变出 **ACID 提交、时间旅行、删除/更新**,让你看清这层魔法的本质其实是**文件级的 MVCC(多版本并发控制)**。
**English**: Data storage long had two split worlds: the **data warehouse** — with ACID transactions, updates/deletes, strict schema, fast queries, but **expensive, closed, hard to scale** (Snowflake, Redshift); and the **data lake** — piling Parquet files on cheap object storage (S3), **cheap, open, infinitely scalable**, but **no transactions, no safe updates/deletes, no versioning, messy schema** (the raw Parquet of 21.8). The **Lakehouse** is the most important architectural revolution of recent years: *add a "transaction log / metadata layer" on top of the cheap data lake (Parquet on S3) and you gain warehouse-grade ACID transactions, updates/deletes, time travel, and schema management*. The magic is realized by three "table formats": **Delta Lake / Apache Iceberg / Apache Hudi**. This section builds a mini-Lakehouse from scratch — real Parquet files + a transaction log — conjuring **ACID commits, time travel, delete/update** by hand, showing that the essence of this magic is really **file-level MVCC (multi-version concurrency control)**.

---

**中文**:**表格式的核心机制**(一句话):**用一个"事务日志"记录"每个版本的表由哪些 Parquet 文件组成"**。
**English**: **The core mechanism of table formats** (one sentence): **use a "transaction log" to record "which Parquet files make up the table at each version."**
- **中文**:**原子提交(atomic commit)**:每次写入(追加/删除/更新)= 往日志里**追加一条记录**,说明"这次新增了哪些文件、移除了哪些文件"。日志追加是原子的,所以**要么整批可见、要么完全不可见**——这就是 ACID 里的原子性。
  **Atomic commit**: each write (append/delete/update) = **append one entry** to the log stating "which files were added and which removed." The log append is atomic, so **either the whole batch is visible or none of it is** — the atomicity in ACID.
- **中文**:**时间旅行(time travel)**:要读版本 N 的表,就"重放日志到第 N 条",算出那一刻的**活跃文件集合**,只读这些文件。旧版本的文件不删,所以**任何历史版本都能查**(审计、回滚、可复现)。
  **Time travel**: to read version N, "replay the log up to entry N," compute the **active file set** at that moment, and read only those files. Old files aren't deleted, so **any historical version is queryable** (audit, rollback, reproducibility).
- **中文**:**更新/删除(copy-on-write)**:对象存储上的 Parquet 文件**不能原地改**,所以"删除某些行"= 读出相关文件、写出**不含那些行的新文件**、在日志里记"移除旧文件+新增新文件"。老文件还在,时间旅行仍能看到删除前的状态。
  **Update/delete (copy-on-write)**: Parquet files on object storage **can't be edited in place**, so "delete some rows" = read the relevant files, write **new files without those rows**, and log "remove old files + add new files." The old files remain, so time travel can still see the pre-delete state.
- **中文**:**快照隔离(snapshot isolation)**:读操作总是看到某个一致的版本快照(一组确定的文件),不会读到别人正在写、写了一半的数据——并发读写安全。
  **Snapshot isolation**: reads always see a consistent version snapshot (a fixed set of files), never half-written data from a concurrent writer — safe concurrent read/write.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 现代数据架构必考）**
> **中文**:**湖仓一体(Lakehouse)**=数据湖的低成本开放 + 数据仓库的 ACID/管理能力。**实现**=在 Parquet(对象存储)之上加**表格式**(Delta/Iceberg/Hudi):一个**事务日志/元数据层**记录每版本由哪些文件组成→带来:①**ACID 事务**(原子提交=原子追加日志)②**时间旅行**(读任意历史版本, 审计/回滚/复现)③**更新/删除/MERGE**(copy-on-write 或 merge-on-read, 对象存储不能原地改故重写文件)④**schema 演进+强制**⑤**快照隔离**(并发读写安全)。**三者对比**:**Delta**(Databricks 系, 事务日志 _delta_log, 生态与 Spark 深绑, 最流行)；**Iceberg**(Netflix 起源, 开放中立, 引擎无关-Spark/Flink/Trino/Snowflake 都支持, 隐藏分区+快照, 势头最猛)；**Hudi**(Uber 起源, 强于流式 upsert/CDC 增量摄取, merge-on-read)。**Medallion 架构**:bronze(原始)→silver(清洗)→gold(聚合)分层。**vs 裸 Parquet**(21.8):裸 Parquet 无事务/无法安全并发更新/无版本, 表格式补上这些。**vs 数仓**:Lakehouse 开放格式+存算分离+便宜, 数仓更成熟但封闭贵。面试金句:*"Lakehouse 在便宜的 Parquet 数据湖上加一层事务日志(Delta/Iceberg/Hudi), 元数据记录每版本由哪些文件组成, 从而获得 ACID 提交、时间旅行、更新删除、schema 管理——本质是文件级 MVCC; 更新删除靠 copy-on-write 重写文件(对象存储不能原地改), 旧文件保留支持时间旅行; Iceberg 引擎中立势头最猛, Delta 绑 Spark 最流行, Hudi 强于流式 upsert。"*
> **English**: **Lakehouse** = the data lake's low cost & openness + the warehouse's ACID & management. **Implementation** = a **table format** (Delta/Iceberg/Hudi) atop Parquet (object storage): a **transaction log / metadata layer** records which files make up each version → giving: ① **ACID transactions** (atomic commit = atomic log append) ② **time travel** (read any historical version — audit/rollback/reproducibility) ③ **update/delete/MERGE** (copy-on-write or merge-on-read, since object-store files can't be edited in place, so files are rewritten) ④ **schema evolution + enforcement** ⑤ **snapshot isolation** (safe concurrent read/write). **The three compared**: **Delta** (Databricks, transaction log `_delta_log`, deeply tied to Spark, most popular); **Iceberg** (from Netflix, open and neutral, engine-agnostic — Spark/Flink/Trino/Snowflake all support it, hidden partitioning + snapshots, strongest momentum); **Hudi** (from Uber, strong at streaming upsert/CDC incremental ingestion, merge-on-read). **Medallion architecture**: bronze (raw) → silver (cleaned) → gold (aggregated) layering. **vs raw Parquet** (21.8): raw Parquet has no transactions / no safe concurrent updates / no versioning; table formats add these. **vs warehouse**: the Lakehouse is open-format + storage-compute-separated + cheap; warehouses are more mature but closed and expensive. Interview line: *"A Lakehouse adds a transaction log (Delta/Iceberg/Hudi) over a cheap Parquet data lake; metadata records which files compose each version, giving ACID commits, time travel, updates/deletes, and schema management — essentially file-level MVCC; updates/deletes use copy-on-write to rewrite files (object stores can't edit in place), and retained old files enable time travel; Iceberg is engine-neutral with the strongest momentum, Delta is Spark-bound and most popular, Hudi excels at streaming upsert."*


In [ ]:

# ============================================================
# 从零实现 mini-Lakehouse:事务日志 over 真实 Parquet 文件 / mini-Lakehouse: transaction log over real Parquet
# 中文:表 = data/ 里的一堆 Parquet 文件 + _log/ 里的提交记录。每次写=原子追加一条日志(记 add/remove 哪些文件)。
# English: a table = Parquet files in data/ + commit entries in _log/. Each write = atomically append a log entry (which files added/removed).
# ============================================================
import os, json, shutil, time, glob, polars as pl
ROOT="/tmp/mini_lake_tbl"
if os.path.exists(ROOT): shutil.rmtree(ROOT)
os.makedirs(f"{ROOT}/_log"); os.makedirs(f"{ROOT}/data")

def commit(add=(), remove=()):                              # 原子提交=往事务日志追加一条 / atomic commit = append a log entry
    ver=len(glob.glob(f"{ROOT}/_log/*.json"))
    json.dump({"version":ver,"add":list(add),"remove":list(remove)}, open(f"{ROOT}/_log/{ver:08d}.json","w"))
    return ver
def snapshot_files(version=None):                           # 重放日志→算出某版本的活跃文件集 / replay log → active file set
    active=set()
    for lg in sorted(glob.glob(f"{ROOT}/_log/*.json")):
        e=json.load(open(lg))
        if version is not None and e["version"]>version: break
        active |= set(e["add"]); active -= set(e["remove"]) # 应用增删 / apply add/remove
    return sorted(active)
def append(df):                                            # 写一个新 Parquet 文件 + 提交 / write a new Parquet + commit
    fn=f"data/part-{int(time.time()*1e6)}.parquet"; df.write_parquet(f"{ROOT}/{fn}"); return commit(add=[fn])
def read(version=None):                                    # 读某版本(默认最新)= 只读其活跃文件 / read a version = its active files
    files=snapshot_files(version)
    return pl.concat([pl.read_parquet(f"{ROOT}/{f}") for f in files]) if files else pl.DataFrame()

# 三次追加 = 三个版本 / three appends = three versions
append(pl.DataFrame({"id":[1,2,3],"city":["NY","LA","SF"]}))   # v0
append(pl.DataFrame({"id":[4,5],  "city":["NY","LA"]}))        # v1
append(pl.DataFrame({"id":[6],    "city":["SF"]}))             # v2
print("当前表(最新版本)/ current table:", read().sort("id").to_dict(as_series=False))
print("时间旅行到 v0 / time travel to v0:", read(0).sort("id").to_dict(as_series=False))
print("时间旅行到 v1 / time travel to v1:", read(1).sort("id").to_dict(as_series=False))


In [ ]:

# ============================================================
# DELETE(copy-on-write)+ 时间旅行回到删除前 / DELETE via copy-on-write + time travel to before it
# 中文:对象存储上 Parquet 不能原地改, 所以删除=重写不含目标行的新文件, 日志记"移除旧+新增新"。旧文件保留→历史可查。
# English: object-store Parquet can't be edited in place, so DELETE = rewrite new files without target rows, log "remove old + add new". Old files kept → history queryable.
# ============================================================
def delete_where(col, val):
    add, rem = [], []
    for f in snapshot_files():
        df=pl.read_parquet(f"{ROOT}/{f}"); kept=df.filter(pl.col(col)!=val)
        if len(kept) < len(df):                            # 该文件有要删的行 → 重写 / this file has rows to delete → rewrite
            nf=f"data/part-{int(time.time()*1e6)}-{len(add)}.parquet"; kept.write_parquet(f"{ROOT}/{nf}")
            add.append(nf); rem.append(f)                  # 新文件替换旧文件 / new file replaces old
    return commit(add=add, remove=rem)                     # 一次原子提交 / one atomic commit

v=delete_where("city","NY")                                # 删除所有 NY 行 / delete all NY rows
print(f"DELETE city='NY' 后(版本 v{v})/ after DELETE:", read().sort("id").to_dict(as_series=False))
print("时间旅行回到 v2(删除前)/ time travel to v2 (before delete):", read(2).sort("id").to_dict(as_series=False))
print("\n事务日志(每条=一次原子提交)/ transaction log (each = one atomic commit):")
for lg in sorted(glob.glob(f"{ROOT}/_log/*.json")):
    e=json.load(open(lg)); print(f"  v{e['version']}: +{len(e['add'])} 文件, -{len(e['remove'])} 文件")
print("→ ACID 提交 + 时间旅行 + 删除, 全靠这个'事务日志记录每版本由哪些文件组成'实现(文件级 MVCC)")


In [ ]:

# ============================================================
# 可视化:事务日志与版本快照 / transaction log & version snapshots
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 每版本的活跃文件数 / active file count per version
vers=list(range(len(glob.glob(f"{ROOT}/_log/*.json"))))
nfiles=[len(snapshot_files(v)) for v in vers]; nrows=[len(read(v)) for v in vers]
ax[0].bar([f"v{v}" for v in vers], nrows, color="#4C72B0")
for i,(nf,nr) in enumerate(zip(nfiles,nrows)): ax[0].text(i,nr+0.1,f"{nr}行\n{nf}文件",ha="center",fontsize=8)
ax[0].set_ylabel("表的行数"); ax[0].set_title("每个版本的表(时间旅行可读任意版本)")
ax[0].annotate("v3: DELETE NY\n(copy-on-write 重写文件)",xy=(3,nrows[3]),xytext=(1.4,5.5),fontsize=8,
               arrowprops=dict(arrowstyle="->",color="#C44E52"))
# ② 事务日志→文件快照示意 / log → snapshot
ax[1].axis("off"); ax[1].set_title("事务日志:每次提交记录 add/remove 文件",fontsize=12,weight="bold")
logtxt=["v0: +file_A (id 1,2,3)","v1: +file_B (id 4,5)","v2: +file_C (id 6)","v3: -file_A,-file_B +重写(去掉NY)"]
for i,txt in enumerate(logtxt):
    c="#C44E52" if i==3 else "#55A868"
    ax[1].add_patch(plt.Rectangle((0.05,0.75-i*0.18),0.9,0.13,fc=c,alpha=0.25,transform=ax[1].transAxes))
    ax[1].text(0.08,0.815-i*0.18,txt,fontsize=9,family="monospace",transform=ax[1].transAxes)
ax[1].text(0.5,0.04,"读版本 N = 重放日志到 N → 得到活跃文件集 → 只读这些文件",ha="center",fontsize=9,style="italic",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/big11_viz.png",dpi=80); plt.show()
print("左:每版本是一致的表快照, 可时间旅行; 右:事务日志记录每次提交的文件增删, 是 ACID+时间旅行的核心")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **湖仓一体的"魔法"其实朴素得惊人:就是一个记录文件清单的日志**:很多人以为 Delta/Iceberg 是什么高深的数据库黑科技,但我们几十行就复现了它的核心——**一个事务日志,记录"每个版本的表由哪些 Parquet 文件组成"**。原子提交 = 原子地追加一条日志;时间旅行 = 重放日志到某版本、算出那时的文件集;删除/更新 = 重写文件 + 在日志里记增删。这就是**文件级的 MVCC**。理解了这一点,你会发现 Delta 的 `_delta_log`、Iceberg 的 metadata/manifest、数据库的 MVCC,本质是**同一个思想**:不原地改数据,而是维护"哪些数据构成当前一致视图"的元数据。
2. **它解决的是数据湖的真实痛点,不是炫技**:裸 Parquet 数据湖(21.8)看似便宜自由,但生产中有致命问题:①**没有原子性**——一个写作业中途失败,会留下**一半的文件**,读到脏数据;②**不能安全更新/删除**——GDPR 要求删某用户数据,裸 Parquet 只能重写整个目录、且过程中读会出错;③**没有版本**——覆盖就永久丢失,无法回滚/审计;④**并发读写打架**。表格式用那个小小的事务日志,一次性解决了这全部问题——这就是为什么它成了现代数据平台的标准底座(而不是继续用裸 Parquet)。
3. **诚实的复杂性与选型**:①**copy-on-write 的代价**:删一行要重写整个文件(甚至因为要保留历史,存储会膨胀)——所以需要定期 **compaction(合并小文件)** 和 **vacuum(清理过期旧版本)**,否则小文件爆炸、成本失控。Hudi 的 merge-on-read 用增量日志缓解写放大,但读时要合并、更复杂——这是**写性能 vs 读性能**的经典权衡。②**三者的现实选择**:**Delta** 最流行但和 Databricks/Spark 绑得深;**Iceberg** 是**开放中立**的赢家——引擎无关(Spark/Flink/Trino/Snowflake/BigQuery 都支持),避免厂商锁定,势头最猛,很多公司正在标准化到 Iceberg;**Hudi** 在**流式 upsert/CDC 增量摄取**场景最强。没有绝对最优,看你的引擎生态和工作负载。③**它不是数据库**:Lakehouse 优化的是"大规模分析扫描 + 批量更新",不是高并发的单行事务(那还是 OLTP 数据库的活)。**结论:Lakehouse(表格式 over Parquet)是"数据湖有了数据库的可靠性"的关键一跃,理解它是文件级 MVCC、掌握 ACID/时间旅行/copy-on-write 的机制、知道 Delta/Iceberg/Hudi 的取舍,是现代数据工程师的核心竞争力。**

**English**:
1. **The Lakehouse "magic" is astonishingly plain: just a log recording file lists**: many think Delta/Iceberg is deep database black magic, but we reproduced its core in a few dozen lines — **a transaction log recording "which Parquet files make up the table at each version."** Atomic commit = atomically append a log entry; time travel = replay the log to a version and compute the file set then; delete/update = rewrite files + log the add/remove. This is **file-level MVCC**. Grasp this and you see that Delta's `_delta_log`, Iceberg's metadata/manifests, and a database's MVCC are the **same idea**: don't edit data in place, but maintain metadata about "which data constitutes the current consistent view."
2. **It solves real data-lake pains, not showing off**: a raw Parquet data lake (21.8) seems cheap and free, but has fatal production problems: ① **no atomicity** — a write job failing midway leaves **half the files**, so readers see dirty data; ② **no safe update/delete** — GDPR requires deleting a user's data, but raw Parquet can only rewrite the whole directory, and reads during it error out; ③ **no versioning** — overwrite loses data forever, no rollback/audit; ④ **concurrent read/write conflicts**. Table formats solve all of this at once with that tiny transaction log — which is why they became the standard foundation of modern data platforms (instead of continuing with raw Parquet).
3. **Honest complexity and tool choice**: ① **copy-on-write's cost**: deleting one row rewrites a whole file (and storage bloats since history is retained) — so you need periodic **compaction (merge small files)** and **vacuum (clean expired old versions)**, else small files explode and costs spiral. Hudi's merge-on-read uses incremental logs to ease write amplification but merges at read time, more complex — the classic **write performance vs read performance** tradeoff. ② **The real choice among the three**: **Delta** is most popular but deeply tied to Databricks/Spark; **Iceberg** is the **open, neutral** winner — engine-agnostic (Spark/Flink/Trino/Snowflake/BigQuery all support it), avoids vendor lock-in, has the strongest momentum, and many companies are standardizing on it; **Hudi** is strongest for **streaming upsert/CDC incremental ingestion**. No absolute best — depends on your engine ecosystem and workload. ③ **It's not a database**: the Lakehouse optimizes "large-scale analytical scans + batch updates," not high-concurrency single-row transactions (still an OLTP database's job). **Conclusion: the Lakehouse (table formats over Parquet) is the key leap to "a data lake with a database's reliability"; understanding it as file-level MVCC, mastering ACID/time-travel/copy-on-write mechanics, and knowing the Delta/Iceberg/Hudi tradeoffs is a core competency for a modern data engineer.**

> 💼 **实战视角 / Practical angle**
> **中文**:湖仓落地:①**新数据平台默认用表格式**(Iceberg 或 Delta)而非裸 Parquet——获得 ACID/时间旅行/更新删除;②**Medallion 分层**:bronze(原始摄取)→silver(清洗/去重/schema 化)→gold(业务聚合), 每层都是表格式表;③**MERGE/upsert** 做 CDC 入湖、慢变维;④**时间旅行**用于审计、回滚坏数据、复现训练数据集(ML 可复现性!);⑤**运维**:定期 compaction 合小文件、vacuum 清旧版本控成本;⑥**GDPR 删除**靠表格式的 DELETE。**选型**:要引擎中立/避免锁定→**Iceberg**(势头最猛); 深度用 Databricks/Spark→**Delta**; 重流式 upsert/CDC→**Hudi**。引擎:Spark/Flink/Trino/DuckDB(delta/iceberg 扩展)都能读。面试金句:*"Lakehouse 在 Parquet 数据湖上加事务日志(Delta/Iceberg/Hudi), 用元数据记录每版本的文件构成实现文件级 MVCC——ACID 提交、时间旅行、copy-on-write 更新删除、schema 管理; 解决裸 Parquet 无事务/无法安全更新/无版本的痛点; 配 Medallion 分层做数据平台; Iceberg 引擎中立最受青睐, Delta 绑 Spark, Hudi 强流式 upsert; 要定期 compaction/vacuum 控成本。"*
> **English**: Lakehouse in practice: ① **use a table format by default for new platforms** (Iceberg or Delta), not raw Parquet — gaining ACID/time-travel/updates-deletes; ② **Medallion layering**: bronze (raw ingest) → silver (clean/dedupe/schematize) → gold (business aggregates), each a table-format table; ③ **MERGE/upsert** for CDC into the lake and slowly-changing dimensions; ④ **time travel** for audit, rolling back bad data, reproducing training datasets (ML reproducibility!); ⑤ **operations**: periodic compaction to merge small files, vacuum to clean old versions and control cost; ⑥ **GDPR deletion** via the table format's DELETE. **Tool choice**: want engine neutrality / avoid lock-in → **Iceberg** (strongest momentum); deep Databricks/Spark use → **Delta**; heavy streaming upsert/CDC → **Hudi**. Engines: Spark/Flink/Trino/DuckDB (delta/iceberg extensions) can all read. Interview line: *"A Lakehouse adds a transaction log (Delta/Iceberg/Hudi) over a Parquet data lake, using metadata about each version's file composition for file-level MVCC — ACID commits, time travel, copy-on-write updates/deletes, schema management; it solves raw Parquet's lack of transactions/safe updates/versioning; pair it with Medallion layering for a data platform; Iceberg is favored for engine neutrality, Delta binds to Spark, Hudi excels at streaming upsert; run periodic compaction/vacuum to control cost."*

---
### 小结 / Summary
- **中文**:Lakehouse=数据湖(便宜开放)+数据仓库(ACID/管理); 靠表格式(Delta/Iceberg/Hudi)在 Parquet 上加事务日志。
- **English**: Lakehouse = data lake (cheap, open) + warehouse (ACID, management); via table formats (Delta/Iceberg/Hudi) adding a transaction log over Parquet.
- **中文**:事务日志记录每版本的文件构成→文件级 MVCC:原子提交、时间旅行、copy-on-write 更新删除、快照隔离。
- **English**: The transaction log records each version's file composition → file-level MVCC: atomic commits, time travel, copy-on-write update/delete, snapshot isolation.
- **中文**:Iceberg 引擎中立势头最猛、Delta 绑 Spark 最流行、Hudi 强流式 upsert; 要 compaction/vacuum 控成本。
- **English**: Iceberg is engine-neutral with strongest momentum, Delta is Spark-bound and most popular, Hudi excels at streaming upsert; run compaction/vacuum to control cost.
